# Preprocessing & Feature Engineering

Load the raw CSV, clean column names, cast numeric types, drop NaNs, create engineered features (average earnings, earnings_per_view, upload_freq, sub_to_earnings, CPM, year/month start), and save the cleaned data as CSV for downstream notebooks.

In [227]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
import matplotlib.pyplot as plt
# import seaborn as sns



In [228]:
# 1. Load raw CSV with proper encoding
csv_path = '../data/Global YouTube Statistics.csv'
df_raw = pd.read_csv(csv_path, encoding='latin-1')
df_raw.info()



<class 'pandas.DataFrame'>
RangeIndex: 1006 entries, 0 to 1005
Data columns (total 29 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   rank                                     1006 non-null   int64  
 1   Youtuber                                 1006 non-null   str    
 2   subscribers                              1003 non-null   float64
 3   video views                              1006 non-null   float64
 4   category                                 951 non-null    str    
 5   Title                                    1006 non-null   str    
 6   uploads                                  1006 non-null   int64  
 7   Country of origin                        881 non-null    str    
 8   Country                                  881 non-null    str    
 9   Abbreviation                             881 non-null    str    
 10  channel_type                             974 non-null    st

In [229]:
df_raw.head(5)

,rank,Youtuber,subscribers,video views,category,Title,uploads,Country of origin,Country,Abbreviation,...,subscribers_for_last_30_days,created_year,created_month,created_date,Gross tertiary education enrollment (%),Population,Unemployment rate,Urban_population,Latitude,Longitude
0,1,T-Series,245000000.0,2.280000e+11,Music,T-Series,20082,India,india,IN,...,2000000.0,2006.0,Mar,13.0,28.1,1.366418e+09,5.36,471031528.0,20.593684,78.962880
1,2,YouTube Movies,170000000.0,0.000000e+00,Film & Animation,youtubemovies,1,United States,United States,US,...,NaN,2006.0,NaN,5.0,88.2,3.282395e+08,14.70,270663028.0,37.090240,-95.712891
2,3,MrBeast,166000000.0,2.836884e+10,Entertainment,MrBeast,741,United States,United States,US,...,8000000.0,2012.0,Feb,20.0,88.2,3.282395e+08,14.70,270663028.0,37.090240,-95.712891
3,4,Cocomelon - Nursery Rhymes,162000000.0,1.640000e+11,Education,Cocomelon - Nursery Rhymes,966,United States,United States,US,...,1000000.0,2006.0,Sep,1.0,88.2,3.282395e+08,14.70,270663028.0,37.090240,-95.712891
4,5,SET India,159000000.0,1.480000e+11,Shows,SET India,116536,India,India,IN,...,1000000.0,2006.0,Sep,20.0,28.1,1.366418e+09,5.36,471031528.0,20.593684,78.962880


## Normalisasi nama kolom

In [230]:
# 2. Clean column names (lowercase, underscores)
df_raw.columns = [c.strip().lower().replace(' ', '_') for c in df_raw.columns]


## Hapus Category Youtube

In [231]:
# Hapus baris yang memiliki nilai `video_views` sama dengan 0
# Pastikan kolom numerik; non-numeric akan menjadi NaN
df_raw['video_views'] = pd.to_numeric(df_raw['video_views'], errors='coerce')
before_rows = len(df_raw)
# Filter: hanya simpan baris dengan video_views != 0
df_raw = df_raw[df_raw['video_views'] != 0]
after_rows = len(df_raw)
# Tampilkan ringkasan singkat (menghindari print sesuai kebijakan proyek)
display(pd.DataFrame({'before_rows':[before_rows], 'after_rows':[after_rows], 'removed':[before_rows-after_rows]}))

,before_rows,after_rows,removed
0,1006,997,9


## Hapus Kolom tidak diperlukan 

In [232]:
cols_to_drop = ['gross_tertiary_education_enrollment_(%)', 'unemployment_rate', 'urban_population']
df_raw = df_raw.drop(columns=cols_to_drop)

In [233]:
df_clean = df_raw.copy()

## Cek duplikat

In [234]:
df_clean.duplicated().sum()

np.int64(10)

In [235]:
print(f"Jumlah baris sebelum: {len(df_clean)}")
df_clean = df_clean.drop_duplicates()
print(f"Jumlah baris sesudah: {len(df_clean)}")


Jumlah baris sebelum: 997
Jumlah baris sesudah: 987


## cek Missing Values

In [249]:
df_clean.isnull().sum()

rank                                 0
youtuber                             0
subscribers                          0
video_views                          0
category                             0
title                                0
uploads                              0
country_of_origin                    0
country                              0
abbreviation                         0
channel_type                         0
video_views_rank                     0
country_rank                         0
channel_type_rank                    0
video_views_for_the_last_30_days     0
lowest_monthly_earnings              0
highest_monthly_earnings             0
lowest_yearly_earnings               0
highest_yearly_earnings              0
subscribers_for_last_30_days         0
created_year                         0
created_month                        0
created_date                         0
population                          96
latitude                            96
longitude                

In [237]:
# preview df_clean
df_clean.head(5)


,rank,youtuber,subscribers,video_views,category,title,uploads,country_of_origin,country,abbreviation,...,highest_monthly_earnings,lowest_yearly_earnings,highest_yearly_earnings,subscribers_for_last_30_days,created_year,created_month,created_date,population,latitude,longitude
0,1,T-Series,245000000.0,2.280000e+11,Music,T-Series,20082,India,india,IN,...,9000000.0,6800000.0,108400000.0,2000000.0,2006.0,Mar,13.0,1.366418e+09,20.593684,78.962880
2,3,MrBeast,166000000.0,2.836884e+10,Entertainment,MrBeast,741,United States,United States,US,...,5400000.0,4000000.0,64700000.0,8000000.0,2012.0,Feb,20.0,3.282395e+08,37.090240,-95.712891
3,4,Cocomelon - Nursery Rhymes,162000000.0,1.640000e+11,Education,Cocomelon - Nursery Rhymes,966,United States,United States,US,...,7900000.0,5900000.0,94800000.0,1000000.0,2006.0,Sep,1.0,3.282395e+08,37.090240,-95.712891
4,5,SET India,159000000.0,1.480000e+11,Shows,SET India,116536,India,India,IN,...,7300000.0,5500000.0,87500000.0,1000000.0,2006.0,Sep,20.0,1.366418e+09,20.593684,78.962880
6,7,ýýý Kids Diana Show,112000000.0,9.324704e+10,People & Blogs,ýýý Kids Diana Show,1111,United States,United States,US,...,2900000.0,2200000.0,35100000.0,NaN,2015.0,May,12.0,3.282395e+08,37.090240,-95.712891


### Handle Category, Country dan Channel_type

In [238]:
cat_cols = ['country', 'country_of_origin', 'abbreviation', 'category', 'channel_type']
for col in cat_cols:
    df_clean[col] = df_clean[col].fillna('Unknown')

In [239]:
# 2. Isi country_rank yang kosong berdasarkan median per negara
# Ini lebih akurat daripada median global
df_clean['country_rank'] = df_clean.groupby('country')['country_rank'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Untuk negara yang masih NaN (karena semua YouTuber di negara itu rank-nya NaN)
# atau untuk negara 'Unknown', isi dengan nilai maksimum + 1 (sebagai penanda rank terbawah)
max_rank = df_clean['country_rank'].max()
df_clean['country_rank'] = df_clean['country_rank'].fillna(max_rank + 1)

In [240]:


# 2. Isi channel_type_rank yang kosong dengan MEDIAN dari tipe yang sama
# Contoh: Jika tipe 'Games' kosong rank-nya, isi dengan median rank para gamers lainnya
df_clean['channel_type_rank'] = df_clean.groupby('channel_type')['channel_type_rank'].transform(
    lambda x: x.fillna(x.median())
)

# 3. Penanganan Akhir: Jika masih ada yang NaN (karena semua di grup itu NaN)
# Isi dengan nilai maksimum yang ada + 1 (sebagai peringkat paling buncit)
max_type_rank = df_clean['channel_type_rank'].max()
df_clean['channel_type_rank'] = df_clean['channel_type_rank'].fillna(max_type_rank + 1)

### Handle subscribers

In [241]:
cols_to_fix = ['subscribers',  'subscribers_for_last_30_days', 'video_views_for_the_last_30_days']
for col in cols_to_fix:
    df_clean[col] = df_clean[col].fillna(0)

In [242]:
# 3. Convert numeric columns (handle scientific notation & missing values)
numeric_cols = [
    'subscribers',
    'video_views',
    'uploads',
    'lowest_monthly_earnings',
    'highest_monthly_earnings',
    'lowest_yearly_earnings',
    'highest_yearly_earnings',
    'subscribers_for_last_30_days',
    'video_views_for_the_last_30_days'
]
for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')


### Handle created_month

In [243]:
# 1. Fungsi untuk mengambil modus (nilai tersering), jika kosong tetap NaN
def get_mode(x):
    mode = x.mode()
    return mode.iloc[0] if not mode.empty else None

# 2. Isi created_month berdasarkan modus di tahun (created_year) yang sama
df_clean['created_month'] = df_clean['created_month'].fillna(
    df_clean.groupby('created_year')['created_month'].transform(get_mode)
)

# 3. Jika masih ada sisa (misal tahun tersebut hanya ada 1 baris dan itu NaN)
# maka isi dengan modus global sebagai backup terakhir
if df_clean['created_month'].isnull().any():
    df_clean['created_month'] = df_clean['created_month'].fillna(df_clean['created_month'].mode()[0])

print("Selesai! Nilai kosong pada created_month telah diisi berdasarkan tren tahunnya.")

Selesai! Nilai kosong pada created_month telah diisi berdasarkan tren tahunnya.


### Handle Unlogic Condition

In [253]:
print(f"Jumlah baris setelah dibersihkan (df_clean sebelum): {len(df_clean)}")

# --- TAHAP IDENTIFIKASI (MENGGUNAKAN df_raw) ---

# Kondisi 1: Upload 0 tapi punya views (Data tidak logis)
mask_zero_uploads = (df_clean['uploads'] == 0) & (df_clean['video_views'] > 0)

# Kondisi 2: Views bulanan lebih besar dari total views (Data korup)
mask_views_inconsistent = df_clean['video_views_for_the_last_30_days'] > df_clean['video_views']

# Kondisi 3: Tahun pembuatan sebelum YouTube lahir (< 2005)
mask_invalid_year = df_clean['created_year'] < 2005

# Kondisi 4: Missing values pada kolom krusial (Subscribers)
mask_missing_subs = df_clean['subscribers'].isna()

# --- TAHAP PEMBERSIHAN (MENGHASILKAN df_clean) ---

# Kita buat copy dari df_clean untuk df_clean
df_clean = df_clean.copy()

# Menghapus baris yang memenuhi kriteria tidak valid di atas
# Kita gunakan operator ~ (NOT) untuk mengambil data yang TIDAK bermasalah
df_clean = df_clean[~(mask_zero_uploads | mask_views_inconsistent | mask_invalid_year | mask_missing_subs)]

# Tambahan: Mengisi sisa missing values pada kategori dengan 'Unknown'
cols_to_fill = ['category', 'country', 'channel_type']
for col in cols_to_fill:
    df_clean[col] = df_clean[col].fillna('Unknown')

# --- RINGKASAN HASIL ---
print(f"Jumlah baris awal (df_raw): {len(df_raw)}")
print(f"Jumlah baris setelah dibersihkan (df_clean sesudah): {len(df_clean)}")
print(f"Total baris yang dibuang: {len(df_raw) - len(df_clean)}")

# Menampilkan 5 baris pertama data yang sudah bersih
df_clean.head()

Jumlah baris setelah dibersihkan (df_clean sebelum): 944
Jumlah baris awal (df_raw): 997
Jumlah baris setelah dibersihkan (df_clean sesudah): 944
Total baris yang dibuang: 53


,rank,youtuber,subscribers,video_views,category,title,uploads,country_of_origin,country,abbreviation,...,highest_monthly_earnings,lowest_yearly_earnings,highest_yearly_earnings,subscribers_for_last_30_days,created_year,created_month,created_date,population,latitude,longitude
0,1,T-Series,245000000.0,2.280000e+11,Music,T-Series,20082,India,india,IN,...,9000000.0,6800000.0,108400000.0,2000000.0,2006.0,Mar,13.0,1.366418e+09,20.593684,78.962880
2,3,MrBeast,166000000.0,2.836884e+10,Entertainment,MrBeast,741,United States,United States,US,...,5400000.0,4000000.0,64700000.0,8000000.0,2012.0,Feb,20.0,3.282395e+08,37.090240,-95.712891
3,4,Cocomelon - Nursery Rhymes,162000000.0,1.640000e+11,Education,Cocomelon - Nursery Rhymes,966,United States,United States,US,...,7900000.0,5900000.0,94800000.0,1000000.0,2006.0,Sep,1.0,3.282395e+08,37.090240,-95.712891
4,5,SET India,159000000.0,1.480000e+11,Shows,SET India,116536,India,India,IN,...,7300000.0,5500000.0,87500000.0,1000000.0,2006.0,Sep,20.0,1.366418e+09,20.593684,78.962880
6,7,Kids Diana Show,112000000.0,9.324704e+10,People & Blogs,Kids Diana Show,1111,United States,United States,US,...,2900000.0,2200000.0,35100000.0,0.0,2015.0,May,12.0,3.282395e+08,37.090240,-95.712891


### Clean Nama Youtuber

In [ ]:
import re


# Fungsi untuk membersihkan karakter non-ASCII
def clean_name(text):
    # Cek jika kosong / NaN
    if pd.isna(text):
        return "Unknown"
    
    # Hapus karakter non-ASCII
    cleaned = re.sub(r'[^\x00-\x7F]+', '', str(text)).strip()
    
    # Jika setelah dibersihkan jadi kosong
    return cleaned if cleaned else "Unknown"

# Terapkan ke kolom
df_clean['youtuber'] = df_clean['youtuber'].apply(clean_name)
df_clean['title'] = df_clean['title'].apply(clean_name)


print(df_clean[df_clean['youtuber'].str.contains("Diana")][['youtuber']])

               youtuber
6       Kids Diana Show
155  Diana and Roma ESP
261  Diana and Roma ARA
268   Diana and Roma EN
843          Lady Diana
934  Diana and Roma IND


## Feature Engineering

In [254]:
# 6. Save cleaned data to CSV 

folder_path = '../data/'
file_name = 'Global_YouTube_Statistics_Cleaned.csv'
full_path = os.path.join(folder_path, file_name)

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder {folder_path} baru saja dibuat.")

df_clean.to_csv(full_path, index=False, encoding='utf-8-sig')

print(f"File berhasil disimpan di: {full_path}")

File berhasil disimpan di: ../data/Global_YouTube_Statistics_Cleaned.csv


In [246]:
# 7. Quick sanity check – basic statistics overview
display(df_clean.describe(include='all'))


,rank,youtuber,subscribers,video_views,category,title,uploads,country_of_origin,country,abbreviation,...,highest_monthly_earnings,lowest_yearly_earnings,highest_yearly_earnings,subscribers_for_last_30_days,created_year,created_month,created_date,population,latitude,longitude
count,944.000000,944,9.440000e+02,9.440000e+02,944,944,944.000000,944,944,944,...,9.440000e+02,9.440000e+02,9.440000e+02,9.440000e+02,944.000000,944,944.000000,8.480000e+02,848.000000,848.000000
unique,NaN,944,NaN,NaN,19,941,NaN,49,50,49,...,NaN,NaN,NaN,NaN,NaN,12,NaN,NaN,NaN,NaN
top,NaN,T-Series,NaN,NaN,Entertainment,Like Nastya Vlog,NaN,United States,United States,US,...,NaN,NaN,NaN,NaN,NaN,Jan,NaN,NaN,NaN,NaN
freq,NaN,1,NaN,NaN,225,2,NaN,303,303,303,...,NaN,NaN,NaN,NaN,NaN,95,NaN,NaN,NaN,NaN
mean,501.045551,NaN,2.265148e+07,1.129473e+10,NaN,NaN,9681.164195,NaN,NaN,NaN,...,6.215831e+05,4.660835e+05,7.463353e+06,2.421546e+05,2012.728814,NaN,15.754237,4.353859e+08,26.636702,-13.565799
std,286.922799,NaN,1.675756e+07,1.437960e+10,NaN,NaN,34994.615017,NaN,NaN,NaN,...,1.170880e+06,8.779036e+05,1.406447e+07,5.375700e+05,4.278306,NaN,8.770249,4.764670e+08,20.526411,84.655258
min,1.000000,NaN,0.000000e+00,4.390980e+05,NaN,NaN,1.000000,NaN,NaN,NaN,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2005.000000,NaN,1.000000,2.025060e+05,-38.416097,-172.104629
25%,253.750000,NaN,1.450000e+07,4.387080e+09,NaN,NaN,241.000000,NaN,NaN,NaN,...,6.435000e+04,4.830000e+04,7.720000e+05,0.000000e+00,2009.000000,NaN,8.000000,8.335541e+07,20.593684,-95.712891
50%,502.500000,NaN,1.770000e+07,7.804773e+09,NaN,NaN,807.000000,NaN,NaN,NaN,...,2.361500e+05,1.771000e+05,2.800000e+06,1.000000e+05,2013.000000,NaN,16.000000,3.282395e+08,37.090240,-51.925280
75%,750.250000,NaN,2.430000e+07,1.392057e+10,NaN,NaN,2919.500000,NaN,NaN,NaN,...,6.440000e+05,4.830000e+05,7.725000e+06,2.000000e+05,2016.000000,NaN,23.000000,3.282395e+08,37.090240,78.962880


In [247]:

df_clean.head(5)

,rank,youtuber,subscribers,video_views,category,title,uploads,country_of_origin,country,abbreviation,...,highest_monthly_earnings,lowest_yearly_earnings,highest_yearly_earnings,subscribers_for_last_30_days,created_year,created_month,created_date,population,latitude,longitude
0,1,T-Series,245000000.0,2.280000e+11,Music,T-Series,20082,India,india,IN,...,9000000.0,6800000.0,108400000.0,2000000.0,2006.0,Mar,13.0,1.366418e+09,20.593684,78.962880
2,3,MrBeast,166000000.0,2.836884e+10,Entertainment,MrBeast,741,United States,United States,US,...,5400000.0,4000000.0,64700000.0,8000000.0,2012.0,Feb,20.0,3.282395e+08,37.090240,-95.712891
3,4,Cocomelon - Nursery Rhymes,162000000.0,1.640000e+11,Education,Cocomelon - Nursery Rhymes,966,United States,United States,US,...,7900000.0,5900000.0,94800000.0,1000000.0,2006.0,Sep,1.0,3.282395e+08,37.090240,-95.712891
4,5,SET India,159000000.0,1.480000e+11,Shows,SET India,116536,India,India,IN,...,7300000.0,5500000.0,87500000.0,1000000.0,2006.0,Sep,20.0,1.366418e+09,20.593684,78.962880
6,7,ýýý Kids Diana Show,112000000.0,9.324704e+10,People & Blogs,ýýý Kids Diana Show,1111,United States,United States,US,...,2900000.0,2200000.0,35100000.0,0.0,2015.0,May,12.0,3.282395e+08,37.090240,-95.712891


In [248]:
df_clean.info()


<class 'pandas.DataFrame'>
Index: 944 entries, 0 to 994
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   rank                              944 non-null    int64  
 1   youtuber                          944 non-null    str    
 2   subscribers                       944 non-null    float64
 3   video_views                       944 non-null    float64
 4   category                          944 non-null    str    
 5   title                             944 non-null    str    
 6   uploads                           944 non-null    int64  
 7   country_of_origin                 944 non-null    str    
 8   country                           944 non-null    str    
 9   abbreviation                      944 non-null    str    
 10  channel_type                      944 non-null    str    
 11  video_views_rank                  944 non-null    float64
 12  country_rank            